# Orquestrador Mestre BDE/RFB — v2 operacional

Versão inicial do **notebook único operacional** para Colab. Diferente da v1, esta versão começa a agregar código real dentro do notebook, sem depender apenas de chamadas a scripts externos.

## Contrato desta versão

- Não altera nem apaga os stages existentes.
- Mantém a v1 como histórico.
- Foco comercial: **estabelecimento ativo por CNPJ completo**.
- Uma empresa (`cnpj_basico`) pode ter estabelecimentos ativos e inativos; o filtro mestre é sempre o CNPJ completo ativo.
- O artefato mensal obrigatório é `estabelecimentos_ativos_chave.parquet`.
- Novas etapas serão agregadas neste mestre em versões futuras.


## 1. Parâmetros, pastas e botões SIM/NÃO

Ajuste a competência e marque `True`/`False` para executar cada etapa.

In [ ]:
from pathlib import Path
import os
import sys
import json
import glob
import zipfile
import subprocess
from datetime import datetime

COMPETENCIA = "2026_06"

# Pasta persistente no Drive. No Colab, monte o Drive antes de rodar etapas que gravam dados.
BASE_DIR = Path(f"/content/drive/MyDrive/BDE_RFB/competencias/{COMPETENCIA}")
RAW_DIR = BASE_DIR / "raw"
ZIP_DIR = RAW_DIR / "zip"
RFB_ZIP_DIR = ZIP_DIR / "rfb"
RFB_EMPRESAS_ZIP_DIR = RFB_ZIP_DIR / "empresas"
RFB_ESTABELECIMENTOS_ZIP_DIR = RFB_ZIP_DIR / "estabelecimentos"
RFB_SOCIOS_ZIP_DIR = RFB_ZIP_DIR / "socios"
RFB_SIMPLES_ZIP_DIR = RFB_ZIP_DIR / "simples"
RFB_AUXILIARES_ZIP_DIR = RFB_ZIP_DIR / "auxiliares"
DOMINIOS_ZIP_DIR = RAW_DIR / "dominios_zip"
PARQUET_DIR = BASE_DIR / "parquet"
BRONZE_DIR = PARQUET_DIR / "bronze"
ATIVOS_DIR = PARQUET_DIR / "ativos"
DOMINIOS_DIR = BASE_DIR / "dominios"
DOMINIOS_EXTRAIDOS_DIR = DOMINIOS_DIR / "extraidos"
DOMINIOS_PARQUET_DIR = PARQUET_DIR / "dominios"
LOG_DIR = BASE_DIR / "logs"
MANIFEST_DIR = BASE_DIR / "manifests"
EXPORT_DIR = BASE_DIR / "exportacoes"

ETAPAS = {
    "00_montar_drive": True,
    "01_criar_pastas": True,
    "02_confirmar_zips_dominios": True,
    "03_processar_zips_dominios": True,
    "04_baixar_zips_rfb": True,
    "05_processar_estabelecimentos_bruto": True,
    "06_processar_empresas_bruto": True,
    "07_criar_chave_estabelecimentos_ativos": True,
    "08_criar_estabelecimentos_ativos": True,
    "09_criar_empresas_ativas": True,
    "10_listar_dominios": True,
    "11_enriquecer_dominios_ativos": False,
    "12_simples_ativos": False,
    "13_socios_ativos": False,
    "14_dividas_pgfn_receita_ativas": False,
    "15_regimes_ativos": False,
    "16_duckdb_final": False,
    "17_exportacoes": False,
}

print("Competência:", COMPETENCIA)
print("Base:", BASE_DIR)
print(json.dumps(ETAPAS, indent=2, ensure_ascii=False))


## 2. Utilitários comuns

Funções reutilizadas por todas as etapas: instalação leve de pacote quando faltar, criação de pastas, manifest, checkpoints, localização de arquivos e leitura de CSV da RFB.

> **Correção DuckDB: evitar parâmetros ? em COPY TO; usar duckdb_lit para caminhos.**


In [ ]:
def run(cmd, check=True):
    print("\n$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"Comando falhou ({p.returncode}): {cmd}")
    return p


def ensure_pkg(import_name, pip_name=None):
    try:
        __import__(import_name)
    except ImportError:
        run(f"{sys.executable} -m pip install -q {pip_name or import_name}")


def duckdb_lit(path):
    return "'" + str(path).replace("\\", "/").replace("'", "''") + "'"


def ensure_dirs():
    for p in [
        BASE_DIR, RAW_DIR, ZIP_DIR, RFB_ZIP_DIR, RFB_EMPRESAS_ZIP_DIR,
        RFB_ESTABELECIMENTOS_ZIP_DIR, RFB_SOCIOS_ZIP_DIR, RFB_SIMPLES_ZIP_DIR,
        RFB_AUXILIARES_ZIP_DIR, DOMINIOS_ZIP_DIR, PARQUET_DIR, BRONZE_DIR,
        ATIVOS_DIR, DOMINIOS_DIR, DOMINIOS_EXTRAIDOS_DIR, DOMINIOS_PARQUET_DIR,
        LOG_DIR, MANIFEST_DIR, EXPORT_DIR,
    ]:
        p.mkdir(parents=True, exist_ok=True)
    print("Pastas prontas:", BASE_DIR)


def now_utc():
    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Gravado:", path)


def checkpoint(nome, payload=None):
    payload = payload or {}
    payload.update({"stage": nome, "competencia": COMPETENCIA, "created_at": now_utc()})
    write_json(LOG_DIR / f"{nome}.ok", payload)


def find_files(patterns):
    files = []
    for pattern in patterns:
        files.extend(glob.glob(str(pattern), recursive=True))
    return sorted({Path(f) for f in files if Path(f).is_file()})


def find_required(patterns, label):
    files = find_files(patterns)
    if not files:
        raise FileNotFoundError(f"Nenhum arquivo encontrado para {label}. Padrões: {patterns}")
    print(f"{label}: {len(files)} arquivo(s)")
    for f in files[:20]:
        print("-", f)
    if len(files) > 20:
        print("...")
    return files


def mount_drive_if_colab():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print("Drive não montado via Colab ou ambiente local:", exc)


def read_rfb_csv_from_zip(zip_path, columns, dtype=str, chunksize=250_000):
    import pandas as pd
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.namelist() if not m.endswith('/')]
        if not members:
            raise ValueError(f"ZIP vazio: {zip_path}")
        member = members[0]
        with zf.open(member) as fh:
            yield from pd.read_csv(
                fh,
                sep=';',
                header=None,
                names=columns,
                dtype=dtype,
                encoding='latin1',
                chunksize=chunksize,
                low_memory=False,
            )

if ETAPAS["00_montar_drive"]:
    mount_drive_if_colab()
if ETAPAS["01_criar_pastas"]:
    ensure_dirs()
    checkpoint("01_criar_pastas", {"base_dir": str(BASE_DIR), "dominios_zip_dir": str(DOMINIOS_ZIP_DIR), "rfb_zip_dir": str(RFB_ZIP_DIR)})


## Painel de retomada — saber onde continuar no dia seguinte

No dia seguinte, rode as células de configuração, montagem do Drive, criação de pastas e depois esta célula. Ela lê os checkpoints `.ok` gravados em `LOG_DIR` e mostra quais etapas já concluíram e qual é a próxima etapa provável.

In [ ]:
ORDEM_ETAPAS = [
    ('01_criar_pastas', 'Criar pastas base'),
    ('02_confirmar_zips_dominios', 'Confirmar ZIPs de domínios'),
    ('03_processar_zips_dominios', 'Processar ZIPs de domínios'),
    ('04_baixar_zips_rfb', 'Baixar ZIPs oficiais RFB'),
    ('05_processar_estabelecimentos_bruto', 'Processar Estabelecimentos bruto'),
    ('06_processar_empresas_bruto', 'Processar Empresas bruto'),
    ('07_criar_chave_estabelecimentos_ativos', 'Criar chave de estabelecimentos ativos'),
    ('08_criar_estabelecimentos_ativos', 'Criar estabelecimentos ativos'),
    ('09_criar_empresas_ativas', 'Criar empresas ativas'),
    ('10_listar_dominios', 'Listar artefatos de domínios'),
    ('11_enriquecer_dominios_ativos', 'Enriquecer domínios ativos'),
    ('12_simples_ativos', 'Processar Simples ativos'),
    ('13_socios_ativos', 'Processar Sócios ativos'),
    ('14_dividas_pgfn_receita_ativas', 'Processar Dívidas/PGFN/Receita ativas'),
    ('15_regimes_ativos', 'Processar Regimes ativos'),
    ('16_duckdb_final', 'Gerar DuckDB final'),
    ('17_exportacoes', 'Gerar exportações'),
]


def carregar_checkpoints():
    checkpoints = {}
    if not LOG_DIR.exists():
        return checkpoints
    for path in sorted(LOG_DIR.glob('*.ok')):
        try:
            payload = json.loads(path.read_text(encoding='utf-8'))
        except Exception:
            payload = {'stage': path.stem, 'erro_leitura': True}
        checkpoints[path.stem] = {'path': str(path), 'payload': payload}
    return checkpoints


def painel_retomada():
    checkpoints = carregar_checkpoints()
    print('Pasta de logs/checkpoints:')
    print(LOG_DIR)
    print('\nStatus das etapas:')
    proxima = None
    for stage, descricao in ORDEM_ETAPAS:
        info = checkpoints.get(stage)
        if info:
            created_at = info['payload'].get('created_at', 'sem data')
            print(f'✅ {stage} — {descricao} — concluída em {created_at}')
        else:
            print(f'⏳ {stage} — {descricao} — pendente')
            if proxima is None:
                proxima = (stage, descricao)
    print('\nResumo:')
    print(f'- Checkpoints encontrados: {len(checkpoints)}')
    if proxima:
        print(f'- Próxima etapa provável: {proxima[0]} — {proxima[1]}')
    else:
        print('- Todas as etapas mapeadas têm checkpoint.')
    return checkpoints

# Rode esta chamada quando quiser saber onde continuar.
painel_retomada()


## 3. Entrada obrigatória — ZIPs de domínios

Depois de criar as pastas, coloque manualmente os ZIPs de domínios em:

```text
{BASE_DIR}/raw/dominios_zip
```

O notebook pergunta se você já inseriu os ZIPs. Se responder `sim`, ele valida, extrai e inventaria os arquivos antes de baixar os ZIPs oficiais da RFB.

In [ ]:
DOMINIOS_CONFIRMADOS = False


def confirmar_zips_dominios():
    ensure_dirs()
    print("Pasta para inserir ZIPs de domínios:")
    print(DOMINIOS_ZIP_DIR)
    existentes = find_files([DOMINIOS_ZIP_DIR / '**' / '*.zip'])
    print(f"ZIPs de domínios encontrados agora: {len(existentes)}")
    for f in existentes[:50]:
        print('-', f)
    resposta = input("Você já inseriu os ZIPs de domínios nessa pasta? digite sim ou nao: ").strip().lower()
    ok = resposta in {'sim', 's', 'yes', 'y'}
    if not ok:
        print("Pare aqui, envie/copiei os ZIPs para a pasta acima e rode esta célula novamente.")
        return False
    if not existentes:
        raise FileNotFoundError(f"Você respondeu sim, mas não encontrei ZIPs em {DOMINIOS_ZIP_DIR}")
    checkpoint('02_confirmar_zips_dominios', {'dominios_zip_dir': str(DOMINIOS_ZIP_DIR), 'zip_count': len(existentes)})
    return True

if ETAPAS['02_confirmar_zips_dominios']:
    DOMINIOS_CONFIRMADOS = confirmar_zips_dominios()
else:
    print('Etapa 02 pulada: confirmação de ZIPs de domínios.')
    DOMINIOS_CONFIRMADOS = True


## 4. Processar ZIPs de domínios antes da RFB

Esta etapa valida os ZIPs de domínios, extrai os arquivos para a pasta da competência e gera manifest/checkpoint. O cruzamento com estabelecimentos ativos entra depois, mas o domínio já fica preparado no começo da execução.

In [ ]:
def zip_is_valid(path):
    try:
        with zipfile.ZipFile(path, 'r') as zf:
            return zf.testzip() is None
    except zipfile.BadZipFile:
        return False


def processar_zips_dominios():
    ensure_dirs()
    zips = find_required([DOMINIOS_ZIP_DIR / '**' / '*.zip'], 'ZIPs de domínios')
    resultados = []
    for zp in zips:
        valido = zip_is_valid(zp)
        item = {'zip': str(zp), 'valid': bool(valido), 'members': []}
        if not valido:
            resultados.append(item)
            continue
        destino_zip = DOMINIOS_EXTRAIDOS_DIR / zp.stem
        destino_zip.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zp, 'r') as zf:
            members = [m for m in zf.namelist() if not m.endswith('/')]
            item['members'] = members
            zf.extractall(destino_zip)
        item['extract_dir'] = str(destino_zip)
        resultados.append(item)
        print(f"OK domínio: {zp.name} -> {destino_zip} ({len(item['members'])} arquivo(s))")

    invalidos = [r for r in resultados if not r['valid']]
    payload = {
        'dominios_zip_dir': str(DOMINIOS_ZIP_DIR),
        'extract_dir': str(DOMINIOS_EXTRAIDOS_DIR),
        'zip_count': len(resultados),
        'invalid_zip_count': len(invalidos),
        'items': resultados,
    }
    write_json(MANIFEST_DIR / 'dominios_zip_processados_manifest.json', payload)
    if invalidos:
        raise RuntimeError(f"Existem ZIPs de domínios inválidos: {invalidos}")
    checkpoint('03_processar_zips_dominios', payload)
    return resultados

if ETAPAS['03_processar_zips_dominios']:
    if not DOMINIOS_CONFIRMADOS:
        raise RuntimeError('ZIPs de domínios ainda não confirmados. Rode a etapa anterior e responda sim.')
    processar_zips_dominios()
else:
    print('Etapa 03 pulada: processar ZIPs de domínios.')


## 5. Baixar ZIPs oficiais da RFB

Depois dos domínios, o mestre baixa os ZIPs oficiais da RFB para subpastas por tipo. Se um ZIP válido já existir, ele não baixa novamente.

In [ ]:
RFB_PUBLIC_ROOT = 'https://arquivos.receitafederal.gov.br/public.php/dav/files/gn672Ad4CF8N6TK/Dados/Cadastros/CNPJ'
RFB_COMPETENCIA_URL = COMPETENCIA.replace('_', '-')
RFB_BASE_URL = f'{RFB_PUBLIC_ROOT}/{RFB_COMPETENCIA_URL}'
RFB_FILES = [
    *[f'Empresas{i}.zip' for i in range(10)],
    *[f'Estabelecimentos{i}.zip' for i in range(10)],
    *[f'Socios{i}.zip' for i in range(10)],
    'Cnaes.zip', 'Motivos.zip', 'Municipios.zip', 'Naturezas.zip', 'Paises.zip', 'Qualificacoes.zip', 'Simples.zip',
]


def destino_rfb_zip(file_name):
    if file_name.startswith('Empresas'):
        return RFB_EMPRESAS_ZIP_DIR / file_name
    if file_name.startswith('Estabelecimentos'):
        return RFB_ESTABELECIMENTOS_ZIP_DIR / file_name
    if file_name.startswith('Socios'):
        return RFB_SOCIOS_ZIP_DIR / file_name
    if file_name.startswith('Simples'):
        return RFB_SIMPLES_ZIP_DIR / file_name
    return RFB_AUXILIARES_ZIP_DIR / file_name


def baixar_arquivo_rfb(file_name):
    ensure_pkg('requests')
    import requests
    destino = destino_rfb_zip(file_name)
    destino.parent.mkdir(parents=True, exist_ok=True)
    url = f"{RFB_BASE_URL}/{file_name}"
    if destino.exists() and zip_is_valid(destino):
        print('Já existe válido:', destino)
        return {'file': file_name, 'url': url, 'path': str(destino), 'status': 'already_exists_valid', 'bytes': destino.stat().st_size}
    tmp = destino.with_suffix(destino.suffix + '.part')
    if tmp.exists():
        tmp.unlink()
    print('Baixando:', url)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as fh:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    fh.write(chunk)
    if not zip_is_valid(tmp):
        raise RuntimeError(f'ZIP baixado inválido: {tmp}')
    tmp.replace(destino)
    return {'file': file_name, 'url': url, 'path': str(destino), 'status': 'downloaded_valid', 'bytes': destino.stat().st_size}


def baixar_zips_rfb():
    ensure_dirs()
    resultados = []
    for file_name in RFB_FILES:
        resultados.append(baixar_arquivo_rfb(file_name))
    payload = {
        'base_url': RFB_BASE_URL,
        'expected_count': len(RFB_FILES),
        'valid_count': sum(1 for r in resultados if r['status'] in {'already_exists_valid', 'downloaded_valid'}),
        'items': resultados,
        'rfb_zip_dir': str(RFB_ZIP_DIR),
    }
    write_json(MANIFEST_DIR / 'rfb_download_manifest.json', payload)
    checkpoint('04_baixar_zips_rfb', payload)
    return resultados

if ETAPAS['04_baixar_zips_rfb']:
    baixar_zips_rfb()
else:
    print('Etapa 04 pulada: baixar ZIPs RFB.')


## 6. Stage operacional — Estabelecimentos bruto

Agrega no notebook a lógica essencial para ler ZIPs de Estabelecimentos da RFB e gerar parquet bruto padronizado com CNPJ completo.

In [ ]:
ESTABELECIMENTOS_COLS = [
    "cnpj_basico", "cnpj_ordem", "cnpj_dv", "matriz_filial", "nome_fantasia",
    "situacao_cadastral", "data_situacao_cadastral", "motivo_situacao_cadastral",
    "nome_cidade_exterior", "pais", "data_inicio_atividade", "cnae_primario",
    "cnaes_secundarias", "tipo_logradouro", "logradouro", "numero", "complemento",
    "bairro", "cep", "uf", "municipio", "ddd_1", "telefone_1", "ddd_2", "telefone_2",
    "ddd_fax", "fax", "correio_eletronico", "situacao_especial", "data_situacao_especial",
]


def only_digits_series(s, width):
    return s.fillna('').astype(str).str.replace(r'\D+', '', regex=True).str.zfill(width).str[-width:]


def processar_estabelecimentos_bruto():
    ensure_pkg('pandas')
    ensure_pkg('pyarrow')
    import pandas as pd

    ensure_dirs()
    zips = find_required([RFB_ESTABELECIMENTOS_ZIP_DIR / '*Estabelecimentos*.zip', RAW_DIR / '**' / '*Estabelecimentos*.zip'], 'ZIPs Estabelecimentos')
    out_path = BRONZE_DIR / 'estabelecimentos_unificado.parquet'
    tmp_dir = BRONZE_DIR / '_tmp_estabelecimentos'
    tmp_dir.mkdir(parents=True, exist_ok=True)
    parts = []

    for zi, zp in enumerate(zips, 1):
        print(f"Processando {zi}/{len(zips)}: {zp}")
        for ci, chunk in enumerate(read_rfb_csv_from_zip(zp, ESTABELECIMENTOS_COLS), 1):
            chunk['cnpj_basico'] = only_digits_series(chunk['cnpj_basico'], 8)
            chunk['cnpj_ordem'] = only_digits_series(chunk['cnpj_ordem'], 4)
            chunk['cnpj_dv'] = only_digits_series(chunk['cnpj_dv'], 2)
            chunk['cnpj'] = chunk['cnpj_basico'] + chunk['cnpj_ordem'] + chunk['cnpj_dv']
            chunk = chunk[chunk['cnpj'].str.len() == 14]
            cols = ['cnpj', 'cnpj_basico', 'cnpj_ordem', 'cnpj_dv'] + [c for c in ESTABELECIMENTOS_COLS if c not in {'cnpj_basico', 'cnpj_ordem', 'cnpj_dv'}]
            part = tmp_dir / f'part_{zi:02d}_{ci:04d}.parquet'
            chunk[cols].to_parquet(part, index=False, compression='zstd')
            parts.append(part)

    if not parts:
        raise RuntimeError('Nenhuma partição de estabelecimentos foi gerada.')

    ensure_pkg('duckdb')
    import duckdb
    con = duckdb.connect()
    con.execute(f"""
        COPY (
            SELECT * EXCLUDE(rn)
            FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY cnpj ORDER BY cnpj) AS rn
                FROM read_parquet(?)
            )
            WHERE rn = 1
        ) TO {duckdb_lit(out_path)} (FORMAT PARQUET, COMPRESSION ZSTD)
    """, [str(tmp_dir / '*.parquet')])
    qa = con.execute("""
        SELECT COUNT(*) rows_total,
               COUNT(DISTINCT cnpj) cnpj_unicos,
               SUM(CASE WHEN situacao_cadastral = '02' THEN 1 ELSE 0 END) ativos_02
        FROM read_parquet(?)
    """, [str(out_path)]).fetchdf().to_dict(orient='records')[0]
    con.close()
    payload = {
        'output': str(out_path),
        'rows_total': int(qa['rows_total']),
        'cnpj_unicos': int(qa['cnpj_unicos']),
        'ativos_02': int(qa['ativos_02'] or 0),
    }
    write_json(MANIFEST_DIR / 'estabelecimentos_unificado_manifest.json', payload)
    checkpoint('05_processar_estabelecimentos_bruto', payload)
    return out_path

if ETAPAS['05_processar_estabelecimentos_bruto']:
    processar_estabelecimentos_bruto()
else:
    print('Etapa 05 pulada: estabelecimentos bruto.')


## 7. Stage operacional — Empresas bruto

Agrega no notebook a lógica essencial para ler ZIPs de Empresas da RFB e gerar parquet por `cnpj_basico`.

In [ ]:
EMPRESAS_COLS = [
    "cnpj_basico", "razao_social", "natureza_juridica", "qualificacao_responsavel",
    "capital_social_str", "porte_empresa", "ente_federativo_responsavel",
]


def processar_empresas_bruto():
    ensure_pkg('pandas')
    ensure_pkg('pyarrow')
    import pandas as pd

    ensure_dirs()
    zips = find_required([RFB_EMPRESAS_ZIP_DIR / '*Empresas*.zip', RAW_DIR / '**' / '*Empresas*.zip'], 'ZIPs Empresas')
    out_path = BRONZE_DIR / 'empresas_unificado.parquet'
    tmp_dir = BRONZE_DIR / '_tmp_empresas'
    tmp_dir.mkdir(parents=True, exist_ok=True)
    parts = []

    for zi, zp in enumerate(zips, 1):
        print(f"Processando {zi}/{len(zips)}: {zp}")
        for ci, chunk in enumerate(read_rfb_csv_from_zip(zp, EMPRESAS_COLS), 1):
            chunk['cnpj_basico'] = only_digits_series(chunk['cnpj_basico'], 8)
            chunk = chunk[chunk['cnpj_basico'].str.len() == 8]
            part = tmp_dir / f'part_{zi:02d}_{ci:04d}.parquet'
            chunk.to_parquet(part, index=False, compression='zstd')
            parts.append(part)

    if not parts:
        raise RuntimeError('Nenhuma partição de empresas foi gerada.')

    ensure_pkg('duckdb')
    import duckdb
    con = duckdb.connect()
    con.execute(f"""
        COPY (
            SELECT * EXCLUDE(rn)
            FROM (
                SELECT *, ROW_NUMBER() OVER (PARTITION BY cnpj_basico ORDER BY cnpj_basico) AS rn
                FROM read_parquet(?)
            )
            WHERE rn = 1
        ) TO {duckdb_lit(out_path)} (FORMAT PARQUET, COMPRESSION ZSTD)
    """, [str(tmp_dir / '*.parquet')])
    qa = con.execute("SELECT COUNT(*) rows_total, COUNT(DISTINCT cnpj_basico) cnpj_basico_unicos FROM read_parquet(?)", [str(out_path)]).fetchdf().to_dict(orient='records')[0]
    con.close()
    payload = {
        'output': str(out_path),
        'rows_total': int(qa['rows_total']),
        'cnpj_basico_unicos': int(qa['cnpj_basico_unicos']),
    }
    write_json(MANIFEST_DIR / 'empresas_unificado_manifest.json', payload)
    checkpoint('06_processar_empresas_bruto', payload)
    return out_path

if ETAPAS['06_processar_empresas_bruto']:
    processar_empresas_bruto()
else:
    print('Etapa 06 pulada: empresas bruto.')


## 8. Artefato obrigatório — chave de estabelecimentos ativos

Cria o parquet mensal com apenas estabelecimentos ativos (`situacao_cadastral = '02'`) e as colunas de join para qualquer etapa posterior.

In [ ]:
def parquet_obrigatorio(nome, padroes):
    files = find_files(padroes)
    if not files:
        raise FileNotFoundError(f"Parquet não encontrado: {nome}. Rode a etapa anterior ou copie o parquet para a pasta-base.")
    print(f"{nome} selecionado:", files[0])
    return files[0]


def criar_chave_estabelecimentos_ativos():
    ensure_pkg('duckdb')
    import duckdb

    ensure_dirs()
    estab = parquet_obrigatorio('estabelecimentos_unificado', [BRONZE_DIR / 'estabelecimentos_unificado.parquet', PARQUET_DIR / '**' / '*estabelec*.parquet'])
    out_path = ATIVOS_DIR / 'estabelecimentos_ativos_chave.parquet'

    con = duckdb.connect()
    con.execute(f"""
        COPY (
            SELECT DISTINCT
                cnpj,
                cnpj_basico,
                cnpj_ordem,
                cnpj_dv,
                data_inicio_atividade AS data_abertura
            FROM read_parquet(?)
            WHERE situacao_cadastral = '02'
              AND cnpj IS NOT NULL AND LENGTH(cnpj) = 14
              AND cnpj_basico IS NOT NULL AND LENGTH(cnpj_basico) = 8
              AND cnpj_ordem IS NOT NULL AND LENGTH(cnpj_ordem) = 4
              AND cnpj_dv IS NOT NULL AND LENGTH(cnpj_dv) = 2
        ) TO {duckdb_lit(out_path)} (FORMAT PARQUET, COMPRESSION ZSTD)
    """, [str(estab)])
    qa = con.execute("""
        SELECT COUNT(*) rows_total,
               COUNT(DISTINCT cnpj) cnpj_unicos,
               COUNT(DISTINCT cnpj_basico) cnpj_basico_unicos,
               SUM(CASE WHEN data_abertura IS NULL OR data_abertura = '' THEN 1 ELSE 0 END) data_abertura_vazia
        FROM read_parquet(?)
    """, [str(out_path)]).fetchdf().to_dict(orient='records')[0]
    con.close()

    payload = {
        'input': str(estab),
        'output': str(out_path),
        'regra': "estabelecimento ativo por cnpj completo: situacao_cadastral = '02'",
        'colunas': ['cnpj', 'cnpj_basico', 'cnpj_ordem', 'cnpj_dv', 'data_abertura'],
        'qa': qa,
    }
    write_json(MANIFEST_DIR / 'estabelecimentos_ativos_chave_manifest.json', payload)
    checkpoint('07_criar_chave_estabelecimentos_ativos', payload)
    return out_path

if ETAPAS['07_criar_chave_estabelecimentos_ativos']:
    criar_chave_estabelecimentos_ativos()
else:
    print('Etapa 07 pulada: chave de estabelecimentos ativos.')


## 9. Derivados ativos — estabelecimentos e empresas

Usa a chave de estabelecimentos ativos como filtro mestre. Estabelecimentos filtram por `cnpj`; empresas filtram por `cnpj_basico` presente em pelo menos um estabelecimento ativo.

In [ ]:
def criar_estabelecimentos_ativos():
    ensure_pkg('duckdb')
    import duckdb
    estab = parquet_obrigatorio('estabelecimentos_unificado', [BRONZE_DIR / 'estabelecimentos_unificado.parquet', PARQUET_DIR / '**' / '*estabelec*.parquet'])
    chave = parquet_obrigatorio('estabelecimentos_ativos_chave', [ATIVOS_DIR / 'estabelecimentos_ativos_chave.parquet'])
    out_path = ATIVOS_DIR / 'estabelecimentos_ativos.parquet'
    con = duckdb.connect()
    con.execute(f"""
        COPY (
            SELECT e.*
            FROM read_parquet(?) e
            INNER JOIN read_parquet(?) a ON e.cnpj = a.cnpj
        ) TO {duckdb_lit(out_path)} (FORMAT PARQUET, COMPRESSION ZSTD)
    """, [str(estab), str(chave)])
    qa = con.execute("SELECT COUNT(*) rows_total, COUNT(DISTINCT cnpj) cnpj_unicos FROM read_parquet(?)", [str(out_path)]).fetchdf().to_dict(orient='records')[0]
    con.close()
    payload = {'input_estabelecimentos': str(estab), 'input_chave': str(chave), 'output': str(out_path), 'qa': qa}
    write_json(MANIFEST_DIR / 'estabelecimentos_ativos_manifest.json', payload)
    checkpoint('08_criar_estabelecimentos_ativos', payload)
    return out_path


def criar_empresas_ativas():
    ensure_pkg('duckdb')
    import duckdb
    empresas = parquet_obrigatorio('empresas_unificado', [BRONZE_DIR / 'empresas_unificado.parquet', PARQUET_DIR / '**' / '*empresas*.parquet'])
    chave = parquet_obrigatorio('estabelecimentos_ativos_chave', [ATIVOS_DIR / 'estabelecimentos_ativos_chave.parquet'])
    out_path = ATIVOS_DIR / 'empresas_ativas.parquet'
    con = duckdb.connect()
    con.execute(f"""
        COPY (
            SELECT DISTINCT emp.*
            FROM read_parquet(?) emp
            INNER JOIN (SELECT DISTINCT cnpj_basico FROM read_parquet(?)) a
                ON emp.cnpj_basico = a.cnpj_basico
        ) TO {duckdb_lit(out_path)} (FORMAT PARQUET, COMPRESSION ZSTD)
    """, [str(empresas), str(chave)])
    qa = con.execute("SELECT COUNT(*) rows_total, COUNT(DISTINCT cnpj_basico) cnpj_basico_unicos FROM read_parquet(?)", [str(out_path)]).fetchdf().to_dict(orient='records')[0]
    con.close()
    payload = {'input_empresas': str(empresas), 'input_chave': str(chave), 'output': str(out_path), 'qa': qa}
    write_json(MANIFEST_DIR / 'empresas_ativas_manifest.json', payload)
    checkpoint('09_criar_empresas_ativas', payload)
    return out_path

if ETAPAS['08_criar_estabelecimentos_ativos']:
    criar_estabelecimentos_ativos()
else:
    print('Etapa 08 pulada: estabelecimentos ativos.')

if ETAPAS['09_criar_empresas_ativas']:
    criar_empresas_ativas()
else:
    print('Etapa 09 pulada: empresas ativas.')


## 10. Domínios já existentes

Primeiro passo operacional para domínios: localizar os artefatos já disponíveis na pasta-base ou no repositório clonado. O enriquecimento real entra na próxima versão, sempre filtrado por `estabelecimentos_ativos_chave.parquet`.

In [ ]:
REPO_DIR = Path('/content/dev_db_cnpj')


def listar_dominios_existentes():
    patterns = [
        DOMINIOS_DIR / '**' / '*',
        BASE_DIR / '**' / '*domini*',
        BASE_DIR / '**' / '*domain*',
        REPO_DIR / 'dominios' / '**' / '*',
        Path.cwd() / 'dominios' / '**' / '*',
    ]
    files = find_files(patterns)
    print(f"Artefatos de domínios encontrados: {len(files)}")
    for f in files[:100]:
        print('-', f)
    if len(files) > 100:
        print('...')
    payload = {'arquivos_encontrados': len(files), 'amostra': [str(f) for f in files[:50]]}
    write_json(MANIFEST_DIR / 'dominios_detectados_manifest.json', payload)
    checkpoint('10_listar_dominios', payload)
    return files

if ETAPAS['10_listar_dominios']:
    listar_dominios_existentes()
else:
    print('Etapa 10 pulada: listar domínios.')

if ETAPAS['11_enriquecer_dominios_ativos']:
    print('Reservado: cruzar domínios somente com estabelecimentos ativos.')
else:
    print('Etapa 11 reservada/pulada: enriquecer domínios ativos.')


## 11. Próximas etapas reservadas

As próximas versões agregam os códigos prontos restantes nesta ordem: Simples, Sócios, Dívidas/PGFN/Receita, Regimes, DuckDB final e Exportações. Todas devem ler `estabelecimentos_ativos_chave.parquet` antes de gravar a saída final.

In [ ]:
PROXIMAS_ETAPAS = [
    '12_simples_ativos',
    '13_socios_ativos',
    '14_dividas_pgfn_receita_ativas',
    '15_regimes_ativos',
    '16_duckdb_final',
    '17_exportacoes',
]
for etapa in PROXIMAS_ETAPAS:
    if ETAPAS[etapa]:
        print(f'{etapa}: reservado para implementação na próxima versão operacional.')
    else:
        print(f'{etapa}: pulada/reservada.')

print('\nCheckpoints:')
for f in sorted(LOG_DIR.glob('*.ok')) if LOG_DIR.exists() else []:
    print('-', f)
